In [0]:
# Load silver table
df_silver = spark.table("silver_db.silver_bank_features")

print("Silver table loaded:", df_silver.count(), "rows")
print("Columns:", df_silver.columns)

Silver table loaded: 10000 rows
Columns: ['CERT', 'REPDTE', 'ASSET', 'NETINC', 'LNLSNET', 'DEP', 'EQTOT', 'ID', 'ingestion_timestamp', 'source', 'layer', 'capital_ratio', 'loan_to_deposit_ratio', 'profit_margin', 'deposit_to_asset_ratio', 'leverage_ratio', 'pagerank', 'degree_centrality', 'betweenness_centrality', 'clustering_coefficient', 'failed']


In [0]:
from pyspark.sql.functions import avg, max, min, stddev, col, when

# Aggregate features per bank (one row per bank)
df_gold = df_silver.groupBy("CERT", "failed").agg(
    avg("ASSET").alias("avg_asset"),
    avg("DEP").alias("avg_dep"),
    avg("EQTOT").alias("avg_equity"),
    avg("NETINC").alias("avg_net_income"),
    avg("capital_ratio").alias("avg_capital_ratio"),
    avg("loan_to_deposit_ratio").alias("avg_loan_to_deposit"),
    avg("profit_margin").alias("avg_profit_margin"),
    avg("leverage_ratio").alias("avg_leverage_ratio"),
    avg("deposit_to_asset_ratio").alias("avg_deposit_to_asset"),
    avg("pagerank").alias("pagerank"),
    avg("degree_centrality").alias("degree_centrality"),
    avg("betweenness_centrality").alias("betweenness_centrality"),
    avg("clustering_coefficient").alias("clustering_coefficient"),
    stddev("ASSET").alias("asset_volatility"),
    stddev("NETINC").alias("income_volatility")
)

print("Gold feature table:", df_gold.count(), "banks,", len(df_gold.columns), "features")
display(df_gold.limit(5))

Gold feature table: 108 banks, 17 features


CERT,failed,avg_asset,avg_dep,avg_equity,avg_net_income,avg_capital_ratio,avg_loan_to_deposit,avg_profit_margin,avg_leverage_ratio,avg_deposit_to_asset,pagerank,degree_centrality,betweenness_centrality,clustering_coefficient,asset_volatility,income_volatility
10016,0,13424.829787234043,11815.659574468085,1522.3617021276596,76.76595744680851,0.11307446808510636,0.4623851063829787,0.0058404255319148965,8.863765957446809,0.8805744680851065,0.007655095118760118,0.07476635514018691,0.018155535846875345,0.6428571428571426,2163.9140596045636,36.6731822204191
10018,0,81042.4,67483.46666666666,11210.0,907.0666666666667,0.13781333333333334,0.4298200000000001,0.011186666666666668,7.285093333333334,0.8329066666666668,0.006493343990497554,0.07476635514018691,0.12291817413458678,0.6428571428571429,5164.800615706284,444.8435466008032
1001,0,100158.49242424243,71720.64393939394,10112.795454545454,654.5151515151515,0.0981765151515152,0.9936780303030299,0.006628030303030308,10.541501515151511,0.7483598484848483,0.00792926387412441,0.07476635514018674,0.12394002775821822,0.6428571428571429,26456.25635137581,416.2401223612828
10025,0,356788.1076923077,309820.43076923076,27561.215384615385,2165.9538461538464,0.07633999999999999,0.6035138461538462,0.005315384615384614,13.238898461538458,0.8895030769230772,0.012038401616218357,0.07476635514018691,0.049977719153623146,0.6428571428571433,662344.9525446767,4687.674218198353
10026,0,72838.88,63110.24,6740.36,319.84,0.092652,0.5759479999999999,0.00446,10.853652000000004,0.8661240000000001,0.010052530446428202,0.07476635514018691,0.12212270882973005,0.6428571428571428,3301.989681994781,367.1911037411809


In [0]:
from pyspark.sql.functions import when, col

# Add risk tier based on financial ratios
df_gold = df_gold.withColumn("risk_tier",
    when(
        (col("avg_capital_ratio") < 0.05) | (col("avg_leverage_ratio") > 20), "CRITICAL"
    ).when(
        (col("avg_capital_ratio") < 0.08) | (col("avg_leverage_ratio") > 15), "HIGH"
    ).when(
        (col("avg_capital_ratio") < 0.10) | (col("avg_leverage_ratio") > 10), "MEDIUM"
    ).otherwise("LOW")
)

print("Risk tier distribution:")
df_gold.groupBy("risk_tier").count().show()

Risk tier distribution:
+---------+-----+
|risk_tier|count|
+---------+-----+
|      LOW|   30|
|   MEDIUM|   51|
|     HIGH|   25|
| CRITICAL|    2|
+---------+-----+



In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_db.gold_bank_features")

print("gold_bank_features saved:", spark.table("gold_db.gold_bank_features").count(), "banks")

gold_bank_features saved: 108 banks
